In [ ]:
%sql
USE CATALOG purgo_databricks;

/* SQL-based logic for f_invntry_bal_dly_hist table in Databricks environment */

-- Create the f_invntry_bal_dly_hist table if not exists
CREATE TABLE IF NOT EXISTS purgo_playground.f_invntry_bal_dly_hist (
    g_capture_dt_yyyymmdd TIMESTAMP NOT NULL COMMENT "Capture date, always current",
    g_company_cd STRING NOT NULL COMMENT "Company code hardcoded as '1050'",
    g_company_currency_cd STRING NOT NULL COMMENT "Currency code hardcoded as 'EUR'",
    g_dnsa_dt_yyyymmdd TIMESTAMP COMMENT "Expiration date from source or default",
    g_expiration_dt_yyyymmdd TIMESTAMP DEFAULT CAST("9999-12-31" AS TIMESTAMP) COMMENT "Default expiration date if missing",
    g_item_nbr STRING NOT NULL COMMENT "Item number, unique and alphanumeric",
    g_location_cd STRING NOT NULL COMMENT "Location code, alphanumeric",
    g_location_status_cd STRING DEFAULT "none" COMMENT "Location status hardcoded as 'none'",
    g_lot_nbr STRING DEFAULT "none" COMMENT "Lot number, default 'none'",
    g_lot_status_cd STRING DEFAULT "none" COMMENT "Lot status hardcoded as 'none'",
    g_plant_cd STRING DEFAULT "CAGER" COMMENT "Plant code hardcoded as 'CAGER'",
    g_qty_allocated DECIMAL(10,2) COMMENT "Allocated quantity, subset of available",
    g_qty_available DECIMAL(10,2) COMMENT "Available quantity, subset of mrp nettable",
    g_qty_financial_nettable DECIMAL(10,2) COMMENT "Financial nettable, subset of on hand",
    g_qty_in_transit DECIMAL(10,2) DEFAULT NULL COMMENT "Quantity in transit always NULL",
    g_qty_mrp_nettable DECIMAL(10,2) COMMENT "MRP nettable quantity, subset of on hand",
    g_qty_on_hand DECIMAL(10,2) COMMENT "On hand quantity from source",
    g_qty_shippable DECIMAL(10,2) COMMENT "Shippable quantity, subset of available",
    g_source_system_cd STRING DEFAULT "nav_ger" NOT NULL COMMENT "Source system code hardcoded",
    g_unit_cost_company_currency DECIMAL(10,2) COMMENT "Unit cost in company currency",
    co_key STRING NOT NULL COMMENT "Concatenation of source system and company code",
    plant_key STRING NOT NULL COMMENT "Concatenation of source system and plant code",
    plant_location_key STRING NOT NULL COMMENT "Concatenation of system, plant and location codes",
    prod_key STRING NOT NULL COMMENT "Concatenation of system and item number",
    prod_plant_key STRING NOT NULL COMMENT "Concatenation of system, item and plant codes",
    prod_plant_location_key STRING NOT NULL COMMENT "Concatenation of system, item, plant, location",
    prod_plant_lot_key STRING NOT NULL COMMENT "Concatenation of system, item, plant, lot",
    prod_plant_lot_location_key STRING NOT NULL COMMENT "Concatenation of system, item, plant, lot, location"
);

-- Common Table Expressions for data extraction and transformation
WITH SourceData AS (
    SELECT
        ilew.Item_No_ AS g_item_nbr,
        iqh.Location_Code AS g_location_cd,
        CURRENT_TIMESTAMP() AS g_capture_dt_yyyymmdd, -- Current timestamp for capture date
        "1050" AS g_company_cd,
        "EUR" AS g_company_currency_cd,
        MAX(ilew.Posting_Date) AS g_dnsa_dt_yyyymmdd,
        CASE WHEN TRIM(lot_no_) = "" THEN "none" ELSE lot_no_ END AS g_lot_nbr,
        COALESCE(iqh.qty_on_hand, 0.0) AS g_qty_on_hand,
        MAX(COALESCE(iqh.qty_on_hand, 0.0)) AS g_qty_available, -- Example transformation with maximum qty on hand
        "nav_ger" AS g_source_system_cd,
        CONCAT("nav_ger", "1050") AS co_key,
        CONCAT("nav_ger", "CAGER") AS plant_key,
        CONCAT("nav_ger", "CAGER", iqh.Location_Code) AS plant_location_key,
        CONCAT("nav_ger", ilew.Item_No_) AS prod_key,
        CONCAT("nav_ger", ilew.Item_No_, "CAGER") AS prod_plant_key,
        CONCAT("nav_ger", ilew.Item_No_, "CAGER", iqh.Location_Code) AS prod_plant_location_key,
        CONCAT("nav_ger", ilew.Item_No_, "CAGER", lot_no_) AS prod_plant_lot_key,
        CONCAT("nav_ger", ilew.Item_No_, "CAGER", lot_no_, iqh.Location_Code) AS prod_plant_lot_location_key
    FROM itemledgerentrieswopd ilew
    LEFT OUTER JOIN item_qty_on_hand iqh
        ON TRIM(ilew.Item_No_) = TRIM(iqh.Item_No_)
        AND TRIM(ilew.Location_Code) = TRIM(iqh.Location_Code)
    LEFT OUTER JOIN reservation_entry re
        ON TRIM(ilew.Item_No_) = TRIM(re.Item_No_)
        AND TRIM(ilew.Location_Code) = TRIM(re.Location_Code)
    WHERE re.reservation_status = "0" AND re.source_type = "37"
    GROUP BY ilew.Item_No_, iqh.Location_Code, lot_no_
)

-- Insert statement for f_invntry_bal_dly_hist table with type safety
INSERT INTO purgo_playground.f_invntry_bal_dly_hist
SELECT
    g_capture_dt_yyyymmdd,
    g_company_cd,
    g_company_currency_cd,
    g_dnsa_dt_yyyymmdd,
    g_expiration_dt_yyyymmdd,
    g_item_nbr,
    g_location_cd,
    g_location_status_cd,
    g_lot_nbr,
    g_lot_status_cd,
    g_plant_cd,
    g_qty_allocated,
    TRY_CAST(g_qty_available AS DECIMAL(10,2)) AS g_qty_available, -- Ensure type consistency
    g_qty_financial_nettable,
    g_qty_in_transit,
    g_qty_mrp_nettable,
    TRY_CAST(g_qty_on_hand AS DECIMAL(10,2)) AS g_qty_on_hand, -- Type safety check
    g_qty_shippable,
    g_source_system_cd,
    g_unit_cost_company_currency,
    co_key,
    plant_key,
    plant_location_key,
    prod_key,
    prod_plant_key,
    prod_plant_location_key,
    prod_plant_lot_key,
    prod_plant_lot_location_key
FROM SourceData
WHERE TRIM(g_location_cd) IS NOT NULL -- Validate against null locales
AND TRIM(g_company_cd) = "1050"; -- Ensure hardcoded company code filter
